# Dietary inference

Estimate whether menu items are **likely vegetarian** or **likely vegan** from the text already available for retrieval. These are discovery hints, not verified restaurant claims. They must not be used for allergen safety or strict dietary filtering.

## Load the normalized menu and completed descriptions

The ingestion notebook supplies `menu`. The existing enrichment cache supplies descriptions without calling Gemini again.

In [1]:
%run ./01_menu_api_ingestion.ipynb

from pathlib import Path

from enrichment_cache import EnrichmentCache
from menu_descriptions import load_cached_descriptions, needs_description

enrichment_path = Path("enrichment") / f"{menu.id}.json"
enrichment_cache = EnrichmentCache.load(enrichment_path, menu.id)
description_results = load_cached_descriptions(menu, enrichment_cache, require_all=True)
description_by_item = {result.item_id: result.text for result in description_results}

print(f"Loaded {len(menu.items)} items and {len(description_results)} inferred descriptions")

/Users/hassankhan/projects/business-projects/order-assistant/.venv/lib/python3.13/site-packages/nbformat/validator.py:434: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  _validate(nbdict, ref, version, version_minor, relax_add_props)


Fetched at: 2026-10-08T13:12:02.467896+00:00
[
  {
    "id": "0ebf4875-ea06-4f54-9149-ac060fedcd64",
    "name": "Roses",
    "description": "",
    "image": "",
    "availability": "AVAILABLE",
    "menu_item_order": [
      "6acc2f74-92ef-4ac7-9d08-f48217524a65",
      "ab1ac371-7f2b-4380-b656-b7139473f786",
      "d59dfa4a-7515-4c94-8f1f-3e52736f372c",
      "8c01995c-2d80-4cd2-8e41-9f0c7d03de0c",
      "aa88011f-8e69-4ce3-a778-2d32f6abf82a",
      "d0538aec-5a9d-47ae-a82a-d7721d7bb53f",
      "34343ca2-bc28-4e63-935c-0134ddf6f249"
    ],
    "subgroups_ordered": [],
    "child_category": false,
    "style": {
      "textColour": "",
      "backgroundColour": ""
    },
    "printer_config": {
      "id": "7228510c-8ab3-4ffd-a79f-b163d5787f59",
      "name": "BAR",
      "ip": "192.168.1.38",
      "pos_id": "111bef5a3c762496",
      "pos_name": "Main POS",
      "store_device_id": "2a68d7ca-b3ad-4e8f-9fcd-d84b4fe5a4bb",
      "device_id": null,
      "printer_type": "NETWORK",
     

## Build small, traceable model inputs

Each input contains only the item name, categories, description, and configurable choices. The input hash makes cached results stale whenever this evidence changes.

In [2]:
import hashlib
import json

category_names = {category.id: category.name for category in menu.categories}

def dietary_input(item):
    description = item.description
    if needs_description(item):
        description = description_by_item.get(item.id)

    return {
        "item_id": item.id,
        "name": item.name,
        "categories": [category_names[id] for id in item.category_ids],
        "description": description,
        "choices": [
            f"{group.name}: {', '.join(choice.name for choice in group.choices)}"
            for group in item.variation_groups
        ],
    }

def input_hash(record):
    text = json.dumps(record, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

dietary_inputs = [dietary_input(item) for item in menu.items]
dietary_inputs[:2]

[{'item_id': 'ef02fbf0-0691-48d1-a080-9d2f7c0fcf1d',
  'name': 'Blanco Candidato 250ml',
  'categories': ['Whites'],
  'description': 'A crisp white wine, often light-bodied with refreshing citrus and floral notes, offering a dry and clean finish.',
  'choices': []},
 {'item_id': '4638ae68-a21d-4a8e-8cf1-3af80b141d35',
  'name': 'Pinot Grigio BTL',
  'categories': ['Whites'],
  'description': 'A popular dry white wine, typically light-bodied with bright acidity, often featuring notes of green apple, pear, and a hint of minerality.',
  'choices': []}]

## Infer likely vegetarian and vegan suitability

The model uses three labels: `likely`, `unlikely`, and `unknown`. `Unknown` is the correct answer when the text is ambiguous. Results are saved after every batch so the notebook can resume safely.

In [3]:
import os
import time

from dotenv import load_dotenv
from google import genai

MODEL_ID = "gemini-2.5-flash"
PROMPT_VERSION = "dietary-v1"
BATCH_SIZE = 25
VALID_LABELS = {"likely", "unlikely", "unknown"}
dietary_cache_path = Path("enrichment") / f"{menu.id}.dietary.json"

if dietary_cache_path.exists():
    dietary_cache = json.loads(dietary_cache_path.read_text(encoding="utf-8"))
    if dietary_cache.get("menu_id") != menu.id:
        raise ValueError("dietary cache belongs to a different menu")
else:
    dietary_cache = {"menu_id": menu.id, "results": {}}

def save_cache():
    dietary_cache_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = dietary_cache_path.with_suffix(".tmp")
    temporary_path.write_text(
        json.dumps(dietary_cache, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    temporary_path.replace(dietary_cache_path)

def current_cached_result(record):
    result = dietary_cache["results"].get(record["item_id"])
    if not isinstance(result, dict):
        return None
    if result.get("input_hash") != input_hash(record):
        return None
    if result.get("model_id") != MODEL_ID or result.get("prompt_version") != PROMPT_VERSION:
        return None
    return result

pending = [record for record in dietary_inputs if current_cached_result(record) is None]
print(f"{len(pending)} items need dietary inference")

224 items need dietary inference


In [4]:
def infer_batch(client, records):
    prompt = (
        "Classify every menu item as likely, unlikely, or unknown for vegetarian and "
        "vegan suitability. This is an unverified discovery hint, not a safety claim. "
        "Use unknown when the supplied text is insufficient or choices can materially "
        "change the answer. Do not assess allergens, gluten-free status, halal, kosher, "
        "health, or cross-contamination. Give one short reason based only on the supplied "
        "text. Return exactly one row for every item. Items: "
        + json.dumps(records, ensure_ascii=False)
    )
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config={
            "temperature": 0,
            "response_mime_type": "application/json",
            "response_schema": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "item_id": {"type": "string"},
                        "vegetarian": {"type": "string", "enum": sorted(VALID_LABELS)},
                        "vegan": {"type": "string", "enum": sorted(VALID_LABELS)},
                        "reason": {"type": "string"},
                    },
                    "required": ["item_id", "vegetarian", "vegan", "reason"],
                },
            },
        },
    )
    return response.parsed or json.loads(response.text)

if pending:
    load_dotenv(override=True)
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        raise RuntimeError("GEMINI_API_KEY is required for uncached items")
    client = genai.Client(api_key=api_key)

    for start in range(0, len(pending), BATCH_SIZE):
        batch = pending[start:start + BATCH_SIZE]
        rows = infer_batch(client, batch)
        rows_by_id = {row["item_id"]: row for row in rows}

        for record in batch:
            row = rows_by_id.get(record["item_id"])
            if row is None:
                raise ValueError(f"model omitted item {record['item_id']!r}")
            if row["vegetarian"] not in VALID_LABELS or row["vegan"] not in VALID_LABELS:
                raise ValueError(f"model returned an invalid label for {record['item_id']!r}")
            dietary_cache["results"][record["item_id"]] = {
                "vegetarian": row["vegetarian"],
                "vegan": row["vegan"],
                "reason": row["reason"].strip(),
                "origin": "inferred",
                "model_id": MODEL_ID,
                "prompt_version": PROMPT_VERSION,
                "input_hash": input_hash(record),
            }

        save_cache()
        print(f"Inferred {min(start + BATCH_SIZE, len(pending))}/{len(pending)} pending items")
        if start + BATCH_SIZE < len(pending):
            time.sleep(6)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Inferred 25/224 pending items
Inferred 50/224 pending items
Inferred 75/224 pending items
Inferred 100/224 pending items
Inferred 125/224 pending items
Inferred 150/224 pending items
Inferred 175/224 pending items
Inferred 200/224 pending items
Inferred 224/224 pending items


## Inspect the results

Review likely classifications before using them for ranking. The canonical menu's verified `dietary_attributes` remain unchanged.

In [5]:
dietary_results = dietary_cache["results"]
item_by_id = {item.id: item for item in menu.items}

for item_id, result in list(dietary_results.items())[:20]:
    print(
        item_by_id[item_id].name,
        f"vegetarian={result['vegetarian']}",
        f"vegan={result['vegan']}",
        f"— {result['reason']}",
    )

print(f"Cached dietary suggestions for {len(dietary_results)} items")

Blanco Candidato 250ml vegetarian=likely vegan=unknown — White wine; fining agents not specified.
Pinot Grigio BTL vegetarian=likely vegan=unknown — White wine; fining agents not specified.
Mangolian Fish Curry vegetarian=unlikely vegan=unlikely — Contains Sea Bass Fillet.
Mix Sizzler(for 2 people) vegetarian=unlikely vegan=unlikely — Contains chicken, lamb, king prawns, and yoghurt.
Pint Pepsi Max vegetarian=likely vegan=likely — A cola soft drink.
Jeera rice vegetarian=likely vegan=likely — Basmati rice cooked with cumin seeds.
Orange Juice vegetarian=likely vegan=likely — Beverage made from oranges.
Vegan Tandoori Roti vegetarian=likely vegan=likely — Explicitly named "Vegan Tandoori Roti".
Pint Pepsi vegetarian=likely vegan=likely — A cola soft drink.
Blanco Candidato BTL vegetarian=likely vegan=unknown — White wine; fining agents not specified.
Apple Juice vegetarian=likely vegan=likely — Beverage made from apples.
Pint Kingfisher vegetarian=likely vegan=unknown — Lager beer; fini

In [ ]:
from collections import Counter

print("Vegetarian:", Counter(row["vegetarian"] for row in dietary_results.values()))
print("Vegan:", Counter(row["vegan"] for row in dietary_results.values()))